# Chapter 4: SPARQL Essentials

**Level 1: Foundations** · ⏱ about 3 hours · Dataset: *Recipes & Nutrition*

### What you will learn

1. The anatomy of a SPARQL `SELECT` query and its JSON results
2. **Triple patterns** and how shared variables create **joins**
3. **FILTER**: comparisons, text functions, `IN`, `NOT EXISTS`
4. **OPTIONAL** for data that may be missing
5. **UNION** for "either / or" questions
6. **ORDER BY, LIMIT, OFFSET** and **DISTINCT**
7. The other query forms: **ASK**, **CONSTRUCT**, **DESCRIBE**
8. Turning results into **typed pandas DataFrames**
9. **Hands-on:** keep queries in `.rq` files and run them with a reusable `run_query()` helper, then **Mini-project 1: the Recipe explorer**

Every query in this chapter runs against the Recipes graph built in Chapter 3.

In [1]:
import json
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog

from kg import client

pd.set_option('display.max_colwidth', 80)
conn = client.connect()
GRAPH = 'urn:tutorial:ch04:recipes'

with client.transaction(conn):
    conn.clear(graph_uri=GRAPH)
    conn.add(stardog.content.File(str(Path('..', 'data', 'recipes', 'recipes.ttl'))), graph_uri=GRAPH)

print('Loaded', conn.select(f'SELECT (COUNT(*) AS ?n) FROM <{GRAPH}> {{ ?s ?p ?o }}')['results']['bindings'][0]['n']['value'],
      'triples into', GRAPH)

Loaded 1020 triples into urn:tutorial:ch04:recipes


### Choosing the graph without `FROM`

So far every query said `FROM <graph>`. The SPARQL protocol also lets the **caller** choose the graphs. In pystardog that's `default_graph_uri=`. The query text then stays free of graph names, so the same query works on any graph. We'll use that throughout this chapter with a tiny helper, `q()`. Section 4.9 replaces it with a proper one.

In [2]:
PREFIXES = '''
PREFIX rec:        <http://example.org/recipes#>
PREFIX recipe:     <http://example.org/recipes/recipe/>
PREFIX ingredient: <http://example.org/recipes/ingredient/>
PREFIX category:   <http://example.org/recipes/category/>
PREFIX allergen:   <http://example.org/recipes/allergen/>
PREFIX skos:       <http://www.w3.org/2004/02/skos/core#>
'''

def q(query):
    '''Run a SELECT on the recipes graph; return a DataFrame of string values.'''
    result = conn.select(PREFIXES + query, default_graph_uri=GRAPH)
    cols = result['head']['vars']
    return pd.DataFrame([{c: b[c]['value'] if c in b else None for c in cols}
                         for b in result['results']['bindings']], columns=cols)

## 4.1 Anatomy of a SELECT query

```sparql
PREFIX rec: <http://example.org/recipes#>      ← 1. prefixes: short names for namespaces

SELECT ?name ?minutes                          ← 2. which variables to return
WHERE {                                        ← 3. the pattern to match
  ?recipe a rec:Recipe ;
          rec:name ?name ;
          rec:prepMinutes ?minutes .
}
ORDER BY ?minutes                              ← 4. solution modifiers: order, limit, …
LIMIT 3
```

The `WHERE` block is a **graph pattern**: triples in which some positions are **variables** (`?recipe`, `?name`). Stardog finds every way to fill in the variables so that all the triples exist in the data. Each way is one **result row**.

In [3]:
q('''
SELECT ?name ?minutes
WHERE {
  ?recipe a rec:Recipe ;
          rec:name ?name ;
          rec:prepMinutes ?minutes .
}
ORDER BY ?minutes
LIMIT 3
''')

,name,minutes
0,Pâtes au pesto,15
1,Pesto Pasta,15
2,Spaghetti à la carbonara,20


Pesto Pasta appears twice. The recipe has two names, English and French (Chapter 3), so there are two ways to fill in `?name`. We'll filter by language in 4.3.

### What comes back

Behind the DataFrame is the standard **SPARQL JSON results** format:

In [4]:
raw = conn.select(PREFIXES + 'SELECT ?name ?minutes WHERE { recipe:pad-thai rec:name ?name ; rec:prepMinutes ?minutes }',
                  default_graph_uri=GRAPH)
print(json.dumps(raw, indent=2, ensure_ascii=False))

{
  "head": {
    "vars": [
      "name",
      "minutes"
    ]
  },
  "results": {
    "bindings": [
      {
        "name": {
          "xml:lang": "en",
          "type": "literal",
          "value": "Pad Thai"
        },
        "minutes": {
          "datatype": "http://www.w3.org/2001/XMLSchema#integer",
          "type": "literal",
          "value": "25"
        }
      },
      {
        "name": {
          "xml:lang": "fr",
          "type": "literal",
          "value": "Pad thaï"
        },
        "minutes": {
          "datatype": "http://www.w3.org/2001/XMLSchema#integer",
          "type": "literal",
          "value": "25"
        }
      }
    ]
  }
}


- `head.vars` lists the columns.
- `results.bindings` has one entry per row, and each value records its **type** (`uri`, `literal`, `bnode`), **datatype** and **language**.
- The value is always a **string** (`"25"`). Our simple `q()` keeps it that way; section 4.9 converts it to real Python types.

> 🔁 **Neo4j equivalent:** `MATCH (r:Recipe) RETURN r.name, r.prepMinutes ORDER BY r.prepMinutes LIMIT 3`. SPARQL's `WHERE` pattern plays the role of `MATCH`.

## 4.2 Triple patterns and joins

When two triple patterns share a variable, the results must agree on it. That's a **join**, with no `JOIN` keyword needed. Here `?i` connects the recipe to its ingredients, and each ingredient to its name:

In [5]:
q('''
SELECT ?ingredient ?category
WHERE {
  recipe:pad-thai rec:hasIngredient ?i .      # a fixed subject: only Pad Thai
  ?i rec:name ?ingredient ;                   # ; = same subject (?i), next predicate
     rec:inCategory ?c .
  ?c rec:name ?category .
}
ORDER BY ?category ?ingredient
''')

,ingredient,category
0,Egg,Eggs
1,Lime,Fruit
2,Peanuts,Nuts & Seeds
3,Rice Noodles,Pasta & Noodles
4,Tofu,Plant Protein
5,Fish Sauce,Sauces & Condiments
6,Shrimp,Shellfish
7,Garlic,Vegetables


A pattern can have variables in **any** position, including the predicate. That makes it easy to explore unknown data: "tell me everything about Pad Thai".

In [6]:
q('''
SELECT ?property ?value
WHERE { recipe:pad-thai ?property ?value . }
ORDER BY ?property
''')

,property,value
0,http://example.org/recipes#created,2025-02-21
1,http://example.org/recipes#cuisine,Thai
2,http://example.org/recipes#description,"Stir-fried rice noodles with shrimp, tofu, egg and peanuts."
3,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/shrimp
4,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/rice-noodles
5,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/tofu
6,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/egg
7,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/fish-sauce
8,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/garlic
9,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/peanuts


The ingredient lines show up as blank node IDs, and `hasIngredient` appears 8 times, once per ingredient.

`DISTINCT` removes duplicate rows. Which **cuisines** exist?

In [7]:
q('SELECT DISTINCT ?cuisine WHERE { ?r rec:cuisine ?cuisine } ORDER BY ?cuisine')

,cuisine
0,American
1,Chinese
2,French
3,German
4,Italian
5,Mediterranean
6,Thai


## 4.3 FILTER

`FILTER(condition)` keeps only the rows where the condition is true.

### Comparisons and logic

In [8]:
q('''
SELECT ?name ?cuisine ?minutes
WHERE {
  ?r a rec:Recipe ; rec:name ?name ; rec:cuisine ?cuisine ; rec:prepMinutes ?minutes .
  FILTER(lang(?name) = "en")                                     # English names only
  FILTER(?minutes <= 25 && (?cuisine = "Italian" || ?cuisine = "Thai"))
}
ORDER BY ?minutes
''')

,name,cuisine,minutes
0,Pesto Pasta,Italian,15
1,Spaghetti Carbonara,Italian,20
2,Pad Thai,Thai,25


Two `FILTER`s behave like `&&`. `IN` is shorter than many `||`:

In [9]:
q('''
SELECT ?name ?cuisine
WHERE {
  ?r rec:name ?name ; rec:cuisine ?cuisine .
  FILTER(lang(?name) = "en" && ?cuisine IN ("French", "German", "Chinese"))
}
ORDER BY ?cuisine
''')

,name,cuisine
0,Chicken Stir Fry,Chinese
1,Vegetable Soup,French
2,Moules Marinieres,French
3,Potato Salad,German


### Text functions

| Function | True when |
|---|---|
| `CONTAINS(?s, "x")` | `?s` contains "x" |
| `STRSTARTS(?s, "x")`, `STRENDS(?s, "x")` | starts or ends with "x" |
| `REGEX(?s, "pattern", "i")` | matches a regular expression (`"i"` = ignore case) |
| `LCASE(?s)`, `UCASE(?s)` | (not a test) lower- or upper-cases text, for case-insensitive matching |
| `STRLEN(?s)` | (not a test) length |
| `lang(?s)`, `langMatches(lang(?s), "fr")` | language tag tests |

In [10]:
q('''
SELECT ?ingredient
WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient .
  FILTER(REGEX(?ingredient, "(oil|sauce)$", "i"))       # names ending in oil or sauce
}
ORDER BY ?ingredient
''')

,ingredient
0,Fish Sauce
1,Olive Oil
2,Sesame Oil
3,Soy Sauce


### NOT EXISTS: "without"

`FILTER NOT EXISTS { pattern }` keeps a row only if the inner pattern has **no** match. It's the standard way to ask "recipes **without** …":

In [11]:
q('''
SELECT ?name
WHERE {
  ?r a rec:Recipe ; rec:name ?name .
  FILTER(lang(?name) = "en")
  FILTER NOT EXISTS { ?r rec:hasIngredient/rec:hasAllergen allergen:milk }
}
ORDER BY ?name
''')

,name
0,Chicken Stir Fry
1,Crispy Chicken Tenders
2,Grilled Salmon
3,Lupin Flatbread
4,Pad Thai
5,Potato Salad
6,Thai Green Curry
7,Vegetable Soup


`rec:hasIngredient/rec:hasAllergen` is a **property path**: "follow `hasIngredient`, then `hasAllergen`". It's a shortcut for two triple patterns with a throwaway variable in the middle. Chapter 6 covers paths fully.

`MINUS { … }` gives the same result in most cases. Prefer `NOT EXISTS`: its meaning is clearer when variables differ.

> 🔁 **Neo4j equivalent:** `WHERE NOT (r)-[:HAS_INGREDIENT]->()-[:HAS_ALLERGEN]->(:Allergen {name:'Milk'})`.

## 4.4 OPTIONAL: data that may be missing

A normal pattern **drops** rows that don't match. `OPTIONAL { … }` keeps the row and leaves the variables empty. That's essential in an open-world graph (Chapter 1), where many facts are simply not recorded.

Which ingredients have allergens? We still want ingredients **without** any:

In [12]:
q('''
SELECT ?ingredient ?allergen
WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient ; rec:inCategory category:vegetables .
  OPTIONAL { ?i rec:hasAllergen/rec:name ?allergen }
}
ORDER BY ?ingredient
''')

,ingredient,allergen
0,Carrot,NaN
1,Celery,Celery
2,Garlic,NaN
3,Onion,NaN
4,Potato,NaN
5,Tomato,NaN


Only Celery has an allergen; the others come back with an empty `allergen`.

`BOUND(?x)` tests whether an optional variable got a value, and `COALESCE` supplies a default:

In [13]:
q('''
SELECT ?ingredient (COALESCE(?allergen, "none") AS ?allergen_label) (BOUND(?allergen) AS ?has_allergen)
WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient ; rec:inCategory category:vegetables .
  OPTIONAL { ?i rec:hasAllergen/rec:name ?allergen }
}
ORDER BY DESC(?has_allergen) ?ingredient
''')

,ingredient,allergen_label,has_allergen
0,Celery,Celery,true
1,Carrot,none,false
2,Garlic,none,false
3,Onion,none,false
4,Potato,none,false
5,Tomato,none,false


> ⚠️ **Order matters.** Put `OPTIONAL` blocks **after** the required patterns. An `OPTIONAL` at the start of a `WHERE` block can make the query match far more than intended, and run slowly.

> 🔁 **Neo4j equivalent:** `OPTIONAL MATCH (i)-[:HAS_ALLERGEN]->(a) RETURN i.name, coalesce(a.name, 'none')`.

## 4.5 UNION: either / or

`{ A } UNION { B }` returns the rows of `A` **plus** the rows of `B`. Which ingredients are a problem for someone with a **nut** allergy? Either they're in the *Nuts & Seeds* category, **or** they carry the *Peanuts* or *Tree Nuts* allergen:

In [14]:
q('''
SELECT DISTINCT ?ingredient ?reason
WHERE {
  ?i rec:name ?ingredient .
  {
    ?i rec:inCategory category:nuts-seeds .
    BIND("category: Nuts & Seeds" AS ?reason)
  }
  UNION
  {
    ?i rec:hasAllergen ?a .
    FILTER(?a IN (allergen:peanuts, allergen:tree-nuts))
    ?a rec:name ?allergen_name .
    BIND(CONCAT("allergen: ", ?allergen_name) AS ?reason)
  }
}
ORDER BY ?ingredient ?reason
''')

,ingredient,reason
0,Almonds,allergen: Tree Nuts
1,Almonds,category: Nuts & Seeds
2,Peanuts,allergen: Peanuts
3,Peanuts,category: Nuts & Seeds
4,Pine Nuts,allergen: Tree Nuts
5,Pine Nuts,category: Nuts & Seeds


Each ingredient appears once for each branch it matched. Sesame Oil has a Sesame allergen, which neither branch asked for, so it's missing. That's intended. `BIND(expr AS ?var)` creates a new variable; Chapter 6 covers it.

> 🔁 **Neo4j equivalent:** two `MATCH … RETURN` queries joined by `UNION`.

## 4.6 ORDER BY, LIMIT, OFFSET

- `ORDER BY ?a DESC(?b)` sorts by several keys, ascending by default.
- `LIMIT n` returns at most *n* rows; `OFFSET k` skips the first *k*. Together they **paginate**.

Always combine `LIMIT`/`OFFSET` with `ORDER BY`, otherwise the pages can overlap between calls.

In [15]:
PAGE = '''
SELECT ?ingredient ?protein
WHERE {{ ?i a rec:Ingredient ; rec:name ?ingredient ; rec:proteinPer100g ?protein . }}
ORDER BY DESC(?protein) ?ingredient
LIMIT 5 OFFSET {offset}
'''
for page in range(2):
    print(f'--- page {page + 1}')
    print(q(PAGE.format(offset=page * 5)).to_string(index=False))

--- page 1


     ingredient protein
    Lupin Flour      40
Parmesan Cheese      38
 Chicken Breast      31
     Mozzarella      28
        Peanuts      26
--- page 2


  ingredient protein
      Shrimp      24
     Almonds      21
Cocoa Powder      20
      Salmon      20
    Pancetta      15


### Computed columns

`SELECT` can compute values with `(expression AS ?name)`. Here's the energy of each Pad Thai ingredient line measured in grams:

In [16]:
q('''
SELECT ?ingredient ?grams (ROUND(?grams * ?kcal / 100) AS ?kcal_total)
WHERE {
  recipe:pad-thai rec:hasIngredientLine ?line .
  ?line rec:ingredient ?i ; rec:quantity ?grams ; rec:unit "g" .
  ?i rec:name ?ingredient ; rec:kcalPer100g ?kcal .
}
ORDER BY DESC(?kcal_total)
''')

,ingredient,grams,kcal_total
0,Rice Noodles,200,218
1,Peanuts,30,170
2,Shrimp,150,149
3,Tofu,100,76


> 🔁 **Neo4j equivalent:** `ORDER BY … SKIP 5 LIMIT 5`. Cypher's `SKIP` is SPARQL's `OFFSET`.

## 4.7 ASK, CONSTRUCT and DESCRIBE

| Form | Returns | pystardog |
|---|---|---|
| `SELECT` | A table of variable bindings | `conn.select()` |
| `ASK` | `True` / `False` | `conn.ask()` |
| `CONSTRUCT` | New triples built from a template | `conn.graph()` |
| `DESCRIBE` | The triples about a resource | `conn.graph()` |

### ASK

In [17]:
def ask(query):
    return conn.ask(PREFIXES + query, default_graph_uri=GRAPH)

print('Any Thai recipe under 30 minutes?  ', ask('ASK { ?r rec:cuisine "Thai" ; rec:prepMinutes ?m FILTER(?m < 30) }'))
print('Any recipe with more than 12 servings?', ask('ASK { ?r rec:servings ?s FILTER(?s > 12) }'))

Any Thai recipe under 30 minutes?   True


Any recipe with more than 12 servings? False


### CONSTRUCT

`CONSTRUCT { template } WHERE { pattern }` builds **new triples** for every match. It's how you reshape data, derive shortcuts, or extract a subgraph. Here's a direct `recipe → allergen` link derived through ingredients:

In [18]:
turtle = conn.graph(PREFIXES + '''
CONSTRUCT { ?r rec:containsAllergen ?a }
WHERE { ?r a rec:Recipe ; rec:hasIngredient/rec:hasAllergen ?a . }
''', default_graph_uri=GRAPH).decode()
from rdflib import Graph
print(f'{len(Graph().parse(data=turtle, format="turtle"))} derived triples. First lines:\n')
print('\n'.join(turtle.splitlines()[:8]))

36 derived triples. First lines:

@prefix rec: <http://example.org/recipes#> .
@prefix allergen: <http://example.org/recipes/allergen/> .
@prefix ingredient: <http://example.org/recipes/ingredient/> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix category: <http://example.org/recipes/category/> .

recipe:spaghetti-carbonara rec:containsAllergen allergen:gluten .


The derived triples are **returned, not stored**. Chapter 5 shows how to save them with `INSERT`, and Chapter 8 how the reasoner infers them automatically.

### DESCRIBE

`DESCRIBE <iri>` returns "everything relevant" about a resource. Stardog returns the triples where it's the subject:

In [19]:
print(conn.graph(PREFIXES + 'DESCRIBE ingredient:parmesan-cheese', default_graph_uri=GRAPH).decode())

@prefix rec: <http://example.org/recipes#> .
@prefix allergen: <http://example.org/recipes/allergen/> .
@prefix ingredient: <http://example.org/recipes/ingredient/> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix category: <http://example.org/recipes/category/> .

ingredient:parmesan-cheese a rec:Ingredient ;
   rec:hasAllergen allergen:milk ;
   rec:ingredientId "I012" ;
   rec:name "Parmesan Cheese" ;
   rec:inCategory category:cheese ;
   rec:kcalPer100g 431.0 ;
   rec:proteinPer100g 38.0 .


## 4.8 Results as typed DataFrames

Our `q()` returns **strings**. That bites as soon as you compute:

In [20]:
df = q('SELECT ?name ?minutes WHERE { ?r rec:name ?name ; rec:prepMinutes ?minutes FILTER(lang(?name) = "en") }')
print('dtype:', df.minutes.dtype, '| sum:', df.minutes.sum()[:40], '…  | max:', df.minutes.max())

dtype: str | sum: 1525453020203025252030203540 …  | max: 45


Summing strings **concatenates** them, and `max` is alphabetical. The fix: each JSON value carries its **datatype**, so convert it. rdflib already knows how to turn `"25"^^xsd:integer` into `25`, `"2025-02-21"^^xsd:date` into a `date`, and so on:

In [21]:
from decimal import Decimal
from rdflib import Literal, URIRef

def to_python(v):
    if v['type'] == 'literal' and 'datatype' in v:
        value = Literal(v['value'], datatype=URIRef(v['datatype'])).toPython()
        return float(value) if isinstance(value, Decimal) else value
    return v['value']

result = conn.select(PREFIXES + '''SELECT ?name ?minutes ?created WHERE {
    ?r rec:name ?name ; rec:prepMinutes ?minutes ; rec:created ?created FILTER(lang(?name) = "en") }''',
    default_graph_uri=GRAPH)
typed = pd.DataFrame([{c: to_python(b[c]) for c in b} for b in result['results']['bindings']])
print(typed.dtypes, '\n')
print('sum:', typed.minutes.sum(), '| max:', typed.minutes.max(),
      '| newest:', typed.created.max(), f'({type(typed.created.max()).__name__})')

name          str
minutes     int64
created    object
dtype: object 

sum: 380 | max: 45 | newest: 2025-11-19 (date)


Now numbers are numbers and dates are dates. This conversion goes into the reusable helper next.

## 4.9 Hands-on: queries in `.rq` files with `run_query()`

Inline query strings are fine for learning. In a real project they cause trouble: they're hard to review, can't be run in Studio unchanged, and invite string formatting (a security risk, see Chapter 6).

**The convention for the rest of the tutorial:**

1. Each query lives in its own file, `src/kg/queries/<topic>/<name>.rq`.
2. The file starts with a comment: a one-line description and a `Parameters:` line.
3. The query has **no `FROM`**: the caller passes the graphs.
4. Values come in as **parameters**, never pasted into the text.

The helper module `src/kg/sparql.py`:

| Function | Does |
|---|---|
| `run_query(conn, name_or_text, graphs=…, **params)` | Runs a `SELECT`, returns a **typed** DataFrame |
| `ask(…)`, `construct(…)` | The same for `ASK` and `CONSTRUCT`/`DESCRIBE` (the latter returns an rdflib `Graph`) |
| `load_query(name)` | Reads a `.rq` file |
| `list_queries(folder)` | A catalogue of the available queries |
| `term(value)` | Converts a Python value into a safe SPARQL term |

In [22]:
from kg import sparql

print(Path(sparql.__file__).read_text()[:2400], '\n…')

"""Run SPARQL stored in .rq files and get pandas DataFrames back.

    from kg import sparql

    sparql.run_query(conn, 'recipes/quick_recipes', graphs=G, max_minutes=30)
    sparql.list_queries('recipes')

Queries live in src/kg/queries/<folder>/<name>.rq. They contain no FROM clause: the graphs are
passed with `graphs=` (the SPARQL protocol's default-graph-uri), so the same file works on any
graph and can be pasted into Studio as it is.
"""
import re
from decimal import Decimal
from pathlib import Path

import pandas as pd
import stardog
from rdflib import Graph, Literal, URIRef

QUERY_DIR = Path(__file__).parent / 'queries'
_NAME = re.compile(r'^[\w/-]+$')


def load_query(name: str) -> str:
  """The text of src/kg/queries/<name>.rq."""
  path = QUERY_DIR / f'{name}.rq'
  if not path.exists():
    raise FileNotFoundError(f'No query {name!r}: expected {path}')
  return path.read_text(encoding='utf-8')


def _text(query: str) -> str:
  """A query name ('recipes/quick_recipes') or the

### How parameters work

`run_query(conn, 'recipes/quick_recipes', graphs=GRAPH, max_minutes=20)`:

1. loads `quick_recipes.rq`, which uses the variable `?max_minutes` without giving it a value,
2. converts `20` into the SPARQL term `"20"^^xsd:integer` with `term()`,
3. sends it as a **binding**, so Stardog treats `?max_minutes` as `20` while running the query.

The value never becomes part of the query **text**, so it can't change the query's meaning. Chapter 6 shows what goes wrong with string formatting.

In [23]:
for value in ['Thai', 20, 4.5, True]:
    print(f'{value!r:8} -> {sparql.term(value)}')
print(sparql.term(URIRef('http://example.org/recipes/recipe/pad-thai')))

'Thai'   -> "Thai"
20       -> "20"^^<http://www.w3.org/2001/XMLSchema#integer>
4.5      -> "4.5"^^<http://www.w3.org/2001/XMLSchema#double>
True     -> "true"^^<http://www.w3.org/2001/XMLSchema#boolean>
<http://example.org/recipes/recipe/pad-thai>


Here's one query file, and running it:

In [24]:
print(sparql.load_query('recipes/quick_recipes'))
sparql.run_query(conn, 'recipes/quick_recipes', graphs=GRAPH, max_minutes=20)

# Recipes ready in at most ?max_minutes minutes.
# Parameters: ?max_minutes (integer)
PREFIX rec:        <http://example.org/recipes#>

SELECT ?name ?cuisine ?minutes
WHERE {
  ?recipe a rec:Recipe ; rec:name ?name ; rec:cuisine ?cuisine ; rec:prepMinutes ?minutes .
  FILTER(lang(?name) = "en")
  FILTER(?minutes <= ?max_minutes)
}
ORDER BY ?minutes ?name



,name,cuisine,minutes
0,Pesto Pasta,Italian,15
1,Grilled Salmon,Mediterranean,20
2,Lupin Flatbread,Mediterranean,20
3,Pancakes,American,20
4,Spaghetti Carbonara,Italian,20


The columns are typed now. `minutes` is an integer column, ready for pandas:

In [25]:
df = sparql.run_query(conn, 'recipes/all_recipes', graphs=GRAPH)
print(df.dtypes, '\n')
df.groupby('cuisine').minutes.agg(['count', 'mean', 'max']).sort_values('count', ascending=False)

recipe        str
name          str
cuisine       str
minutes     int64
servings    int64
dtype: object 



,count,mean,max
cuisine,,,
American,3,31.666667,45
Italian,3,21.666667,30
French,2,32.500000,40
Thai,2,30.000000,35
Mediterranean,2,20.000000,20
Chinese,1,25.000000,25
German,1,30.000000,30


To run a `.rq` file in **Stardog Studio**, open it, add `FROM <urn:tutorial:ch04:recipes>` after the `SELECT` line, and replace parameters such as `?max_minutes` with a value, or add `VALUES ?max_minutes { 20 }` (Chapter 6).

## Mini-project 1: the Recipe explorer

Level 1 ends with a small project: answer ten everyday questions about the recipe collection using **only `.rq` files and `run_query()`**. The queries are in `src/kg/queries/recipes/`:

In [26]:
sparql.list_queries('recipes')

,query,description,parameters
0,recipes/all_recipes,"All recipes with cuisine, prep time and servings, quickest first.",none
1,recipes/french_menu,"The menu in French, with the English description where there is one.",none
2,recipes/high_protein_ingredients,Ingredients with at least ?min_protein grams of protein per 100 g.,?min_protein (number)
3,recipes/meat_free_recipes,"Recipes without meat, poultry, pork, fish or shellfish.",none
4,recipes/quick_recipes,Recipes ready in at most ?max_minutes minutes.,?max_minutes (integer)
5,recipes/recipe_allergens,"Every recipe with each allergen it contains, and the ingredient responsible.",none
6,recipes/recipes_by_cuisine,Recipes of one cuisine.,"?cuisine (text, e.g. ""Italian"")"
7,recipes/recipes_with_ingredient,"Recipes that use an ingredient, found by (part of) its name, case-insensitive.","?search (text, e.g. ""cheese"")"
8,recipes/safe_quick_recipes,Recipes ready in at most ?max_minutes that contain no ingredient with the gi...,"?allergen_name (text, e.g. ""Milk""), ?max_minutes (integer)"
9,recipes/shopping_list,Ingredients with quantities for one recipe.,"?recipe (IRI, e.g. recipe:pad-thai)"


Each question is one line of Python:

In [27]:
from IPython.display import Markdown, display

PAD_THAI = URIRef('http://example.org/recipes/recipe/pad-thai')

questions = [
    ('What can I cook in 20 minutes or less?',          'recipes/quick_recipes',            dict(max_minutes=20)),
    ('Which Thai dishes are there?',                     'recipes/recipes_by_cuisine',       dict(cuisine='Thai')),
    ('What do I need to buy for Pad Thai?',              'recipes/shopping_list',            dict(recipe=PAD_THAI)),
    ('Which recipes use cheese?',                        'recipes/recipes_with_ingredient',  dict(search='cheese')),
    ('Dairy-free dishes ready in 25 minutes?',           'recipes/safe_quick_recipes',       dict(allergen_name='Milk', max_minutes=25)),
    ('Which recipes are meat- and fish-free?',           'recipes/meat_free_recipes',        {}),
    ('Which ingredients have 20 g+ protein per 100 g?',  'recipes/high_protein_ingredients', dict(min_protein=20)),
]
for question, query, params in questions:
    args = ', '.join(f'{k}={v}' for k, v in params.items())
    display(Markdown(f'**{question}**  `{query}` {args}'))
    display(sparql.run_query(conn, query, graphs=GRAPH, **params))

**What can I cook in 20 minutes or less?**  `recipes/quick_recipes` max_minutes=20

,name,cuisine,minutes
0,Pesto Pasta,Italian,15
1,Grilled Salmon,Mediterranean,20
2,Lupin Flatbread,Mediterranean,20
3,Pancakes,American,20
4,Spaghetti Carbonara,Italian,20


**Which Thai dishes are there?**  `recipes/recipes_by_cuisine` cuisine=Thai

,name,minutes,description
0,Pad Thai,25,"Stir-fried rice noodles with shrimp, tofu, egg and peanuts."
1,Thai Green Curry,35,"Chicken simmered in coconut milk and green curry paste, served with rice."


**What do I need to buy for Pad Thai?**  `recipes/shopping_list` recipe=http://example.org/recipes/recipe/pad-thai

...

,name,cuisine,minutes
0,Grilled Salmon,Mediterranean,20
1,Lupin Flatbread,Mediterranean,20
2,Chicken Stir Fry,Chinese,25
3,Pad Thai,Thai,25


**Which recipes are meat- and fish-free?**  `recipes/meat_free_recipes` 

,name,cuisine
0,Chocolate Brownies,American
1,Lupin Flatbread,Mediterranean
2,Margherita Pizza,Italian
3,Pancakes,American
4,Pesto Pasta,Italian
5,Potato Salad,German
6,Vegetable Soup,French


**Which ingredients have 20 g+ protein per 100 g?**  `recipes/high_protein_ingredients` min_protein=20

,ingredient,protein,kcal,category
0,Lupin Flour,40.0,371.0,Flour
1,Parmesan Cheese,38.0,431.0,Cheese
2,Chicken Breast,31.0,165.0,Poultry
3,Mozzarella,28.0,280.0,Cheese
4,Peanuts,26.0,567.0,Nuts & Seeds
5,Shrimp,24.0,99.0,Shellfish
6,Almonds,21.0,579.0,Nuts & Seeds
7,Cocoa Powder,20.0,228.0,Sweeteners & Baking
8,Salmon,20.0,208.0,Fish


The last three questions return longer tables:

In [28]:
allergens = sparql.run_query(conn, 'recipes/recipe_allergens', graphs=GRAPH)
display(Markdown(f'**Which allergens does each recipe contain, and why?** ({len(allergens)} rows, first 10)'))
display(allergens.head(10))

display(Markdown('**How often does each allergen occur across recipes?**'))
display(allergens.groupby('allergen').recipe_name.nunique().sort_values(ascending=False).rename('recipes').to_frame().T)

display(Markdown('**The whole menu in French**'))
sparql.run_query(conn, 'recipes/french_menu', graphs=GRAPH)

**Which allergens does each recipe contain, and why?** (40 rows, first 10)

,recipe_name,allergen,because_of
0,Chicken Stir Fry,Gluten,Soy Sauce
1,Chicken Stir Fry,Sesame,Sesame Oil
2,Chicken Stir Fry,Soybeans,Soy Sauce
3,Chocolate Brownies,Eggs,Egg
4,Chocolate Brownies,Gluten,Wheat Flour
5,Chocolate Brownies,Milk,Butter
6,Chocolate Brownies,Tree Nuts,Almonds
7,Crispy Chicken Tenders,Eggs,Egg
8,Crispy Chicken Tenders,Gluten,Bread Crumbs
9,Crispy Chicken Tenders,Gluten,Wheat Flour


**How often does each allergen occur across recipes?**

allergen,Gluten,Eggs,Milk,Fish,Crustaceans,Celery,Tree Nuts,Soybeans,Lupin,Molluscs,Peanuts,Mustard,Sesame,Sulphites
recipes,7,6,6,3,2,2,2,2,1,1,1,1,1,1


**The whole menu in French**

,nom,cuisine,minutes,description
0,Aiguillettes de poulet croustillantes,American,30,"Breaded chicken strips, oven-baked."
1,Brownies au chocolat,American,45,Fudgy chocolate brownies with toasted almonds.
2,Curry vert thaï,Thai,35,"Chicken simmered in coconut milk and green curry paste, served with rice."
3,Galette à la farine de lupin,Mediterranean,20,High-protein flatbread made with lupin flour.
4,Moules marinières,French,25,"Mussels steamed in white wine, garlic and cream."
5,Pad thaï,Thai,25,"Stir-fried rice noodles with shrimp, tofu, egg and peanuts."
6,Pancakes,American,20,Fluffy breakfast pancakes.
7,Pizza Margherita,Italian,30,"Thin-crust pizza with tomato, mozzarella and fresh basil."
8,Poulet sauté au wok,Chinese,25,"Chicken and vegetables stir-fried in soy sauce and sesame oil, served with r..."
9,Pâtes au pesto,Italian,15,Spaghetti with basil and pine nut pesto.


The allergen frequency table was computed in **pandas** from the query result. Chapter 6 does the same in SPARQL with `GROUP BY`, which is better when the data is large.

## 4.10 Exercises

**Exercise 1.** List the **Italian** recipes with their number of servings, sorted by servings (most first), then name. English names only.

<details><summary>Solution</summary>

```python
q('''SELECT ?name ?servings WHERE {
  ?r rec:cuisine "Italian" ; rec:name ?name ; rec:servings ?servings .
  FILTER(lang(?name) = "en") }
ORDER BY DESC(?servings) ?name''')
```
</details>

**Exercise 2.** List every **ingredient** with its **category** and, where the category has one, its **parent category** (`skos:broader`). Ingredients whose category has no parent must still appear.

<details><summary>Solution</summary>

```python
q('''SELECT ?ingredient ?category ?parent WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient ; rec:inCategory ?c .
  ?c rec:name ?category .
  OPTIONAL { ?c skos:broader/rec:name ?parent }
} ORDER BY ?parent ?category ?ingredient''')
```
</details>

**Exercise 3.** Create a new query file `src/kg/queries/recipes/recipes_by_allergen.rq`: recipes that **do** contain a given allergen (parameter `?allergen_name`). Run it with `run_query()` for `"Gluten"`.

<details><summary>Solution</summary>

File content:
```sparql
# Recipes that contain an ingredient with the given allergen.
# Parameters: ?allergen_name (text, e.g. "Gluten")
PREFIX rec: <http://example.org/recipes#>

SELECT DISTINCT ?name
WHERE {
  ?recipe a rec:Recipe ; rec:name ?name ;
          rec:hasIngredient/rec:hasAllergen/rec:name ?allergen_name .
  FILTER(lang(?name) = "en")
}
ORDER BY ?name
```
```python
sparql.run_query(conn, 'recipes/recipes_by_allergen', graphs=GRAPH, allergen_name='Gluten')
```
</details>

**Exercise 4.** Translate this Cypher to SPARQL:

```cypher
MATCH (r:Recipe)-[:HAS_INGREDIENT]->(i:Ingredient)
WHERE i.kcalPer100g > 500
RETURN DISTINCT r.name, i.name ORDER BY r.name
```

<details><summary>Solution</summary>

```python
q('''SELECT DISTINCT ?recipe ?ingredient WHERE {
  ?r a rec:Recipe ; rec:name ?recipe ; rec:hasIngredient ?i .
  ?i rec:name ?ingredient ; rec:kcalPer100g ?kcal .
  FILTER(?kcal > 500 && lang(?recipe) = "en")
} ORDER BY ?recipe''')
```
</details>

In [29]:
# your answers here

## 4.11 Clean up

In [30]:
CLEAN_UP = True

if CLEAN_UP:
    with client.transaction(conn):
        conn.clear(graph_uri=GRAPH)
    print('Removed', GRAPH)
conn.close()

Removed urn:tutorial:ch04:recipes


## Summary

- A `SELECT` query matches a **graph pattern**; each way of filling in the variables is a row. Shared variables are **joins**.
- **FILTER** narrows rows with comparisons, `IN`, text functions and `NOT EXISTS` ("without …").
- **OPTIONAL** keeps rows when data is missing (open world); `BOUND` and `COALESCE` handle the gaps.
- **UNION** answers "either / or"; **ORDER BY / LIMIT / OFFSET** sort and paginate; **DISTINCT** removes duplicates.
- **ASK** answers yes/no; **CONSTRUCT** builds new triples; **DESCRIBE** fetches what's known about a resource.
- SPARQL JSON values are strings with a datatype; convert them to get **typed DataFrames**.
- From now on, queries live in **`.rq` files** with no `FROM`, and run through **`sparql.run_query(conn, name, graphs=…, **params)`**.

### SPARQL ↔ Cypher cheat sheet

| SPARQL | Cypher |
|---|---|
| `?r rec:hasIngredient ?i` | `(r)-[:HAS_INGREDIENT]->(i)` |
| `FILTER(?m <= 20)` | `WHERE r.minutes <= 20` |
| `FILTER NOT EXISTS { … }` | `WHERE NOT (…)` |
| `OPTIONAL { … }` | `OPTIONAL MATCH …` |
| `{ … } UNION { … }` | `… UNION …` |
| `ORDER BY DESC(?x) LIMIT 5 OFFSET 10` | `ORDER BY x DESC SKIP 10 LIMIT 5` |
| bindings `?max_minutes` | parameters `$max_minutes` |

🎉 **Level 1 complete.** You can connect, load RDF, and query it from Python in a maintainable way.

**Next: Chapter 5, Updates and transactions**, which opens Level 2. It covers changing data with SPARQL Update, transaction semantics, batching, and an idempotent upsert.